In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
from datasets import load_dataset

conll = load_dataset("conll2003")
conll = conll.rename_column("ner_tags", "labels")

ontonotes = load_dataset("tner/ontonotes5")
ontonotes = ontonotes.rename_column("tags", "labels")

E:\Software\Anaconda\envs\gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

ontonotes_id_to_label = {
    0: "O", 1: "B-CARDINAL", 2: "B-DATE", 3: "I-DATE", 4: "B-PERSON", 5: "I-PERSON",
    6: "B-NORP", 7: "B-GPE", 8: "I-GPE", 9: "B-LAW", 10: "I-LAW", 11: "B-ORG", 12: "I-ORG",
    13: "B-PERCENT", 14: "I-PERCENT", 15: "B-ORDINAL", 16: "B-MONEY", 17: "I-MONEY",
    18: "B-WORK_OF_ART", 19: "I-WORK_OF_ART", 20: "B-FAC", 21: "B-TIME", 22: "I-CARDINAL",
    23: "B-LOC", 24: "B-QUANTITY", 25: "I-QUANTITY", 26: "I-NORP", 27: "I-LOC",
    28: "B-PRODUCT", 29: "I-TIME", 30: "B-EVENT", 31: "I-EVENT", 32: "I-FAC",
    33: "B-LANGUAGE", 34: "I-PRODUCT", 35: "I-ORDINAL", 36: "I-LANGUAGE"
}

ontonotes_label_to_id = {v: k for k, v in ontonotes_id_to_label.items()}

ontonotes_labels = [ontonotes_id_to_label[i]
                    for i in range(len(ontonotes_id_to_label))]

In [4]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [5]:
conll_label_to_id = {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3,
                     'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}

In [6]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",  # or your chosen model
    num_labels=len(conll_label_to_id)  # must match the remapped label space
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True, padding=True)

    all_labels = []
    for i, labels in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

In [8]:
from datasets import concatenate_datasets

# Combine CoNLL splits
conll_combined = concatenate_datasets([
    conll["train"],
    conll["validation"],
    conll["test"]
])

# Combine OntoNotes splits
ontonotes_combined = concatenate_datasets([
    ontonotes["train"],
    ontonotes["validation"],
    ontonotes["test"]
])

In [9]:
print(f"CoNLL Combined: {len(conll_combined)} examples")
print(f"OntoNotes Combined: {len(ontonotes_combined)} examples")

CoNLL Combined: 20744 examples
OntoNotes Combined: 76714 examples


In [10]:

ontonotes_id_to_label = {
    0: "O", 1: "B-CARDINAL", 2: "B-DATE", 3: "I-DATE", 4: "B-PERSON", 5: "I-PERSON",
    6: "B-NORP", 7: "B-GPE", 8: "I-GPE", 9: "B-LAW", 10: "I-LAW", 11: "B-ORG", 12: "I-ORG",
    13: "B-PERCENT", 14: "I-PERCENT", 15: "B-ORDINAL", 16: "B-MONEY", 17: "I-MONEY",
    18: "B-WORK_OF_ART", 19: "I-WORK_OF_ART", 20: "B-FAC", 21: "B-TIME", 22: "I-CARDINAL",
    23: "B-LOC", 24: "B-QUANTITY", 25: "I-QUANTITY", 26: "I-NORP", 27: "I-LOC",
    28: "B-PRODUCT", 29: "I-TIME", 30: "B-EVENT", 31: "I-EVENT", 32: "I-FAC",
    33: "B-LANGUAGE", 34: "I-PRODUCT", 35: "I-ORDINAL", 36: "I-LANGUAGE"
}

ontonotes_to_conll_entity = {
    "PERSON": "PER", "ORG": "ORG", "GPE": "LOC", "LOC": "LOC",
    "NORP": "MISC", "FAC": "MISC", "EVENT": "MISC", "WORK_OF_ART": "MISC",
    "LAW": "MISC", "PRODUCT": "MISC", "LANGUAGE": "MISC",
    "DATE": None, "TIME": None, "PERCENT": None, "MONEY": None,
    "QUANTITY": None, "ORDINAL": None, "CARDINAL": None
}

bio_ontonotes_to_conll = {"O": "O"}
for tag, mapped in ontonotes_to_conll_entity.items():
    if mapped:
        bio_ontonotes_to_conll[f"B-{tag}"] = f"B-{mapped}"
        bio_ontonotes_to_conll[f"I-{tag}"] = f"I-{mapped}"
    else:
        bio_ontonotes_to_conll[f"B-{tag}"] = None
        bio_ontonotes_to_conll[f"I-{tag}"] = None


def map_ontonotes_labels_to_conll_soft(label_id):
    # print('label id', label_id)
    original_label = ontonotes_id_to_label[label_id]
    # print('og label', original_label)
    mapped_label = bio_ontonotes_to_conll.get(original_label)
    # print('mapped label', mapped_label)
    # print()
    if mapped_label is None:
        return conll_label_to_id["O"]
    return conll_label_to_id[mapped_label]


def remap_labels(example):
    # print(example)
    example["labels"] = [map_ontonotes_labels_to_conll_soft(
        label_id) for label_id in example["labels"]]
    return example


ontonotes_combined = ontonotes_combined.map(remap_labels)
# subset = subset.map(remap_labels)

In [11]:
conll_encoded = conll_combined.map(tokenize_and_align_labels, batched=True)

In [12]:
ontonotes_encoded = ontonotes_combined.map(
    tokenize_and_align_labels, batched=True)

In [13]:
id2label = {v: k for k, v in conll_label_to_id.items()}

In [14]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate

metric = evaluate.load("seqeval")


def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=-1)

    true_predictions = [
        [id2label[pred]
            for pred, label in zip(prediction, label_row) if label != -100]
        for prediction, label_row in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[label]
            for pred, label in zip(prediction, label_row) if label != -100]
        for prediction, label_row in zip(predictions, labels)
    ]

    return metric.compute(predictions=true_predictions, references=true_labels)


args = TrainingArguments(
    output_dir="./results",
    eval_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    report_to="none",
    fp16=True,
    logging_steps=1000,
    save_strategy="no",
)

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=conll_encoded,
    eval_dataset=ontonotes_encoded,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [17]:
trainer.train()

E:\Software\Anaconda\envs\gpu_env\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Step,Training Loss
1000,0.149900
2000,0.062800
3000,0.042300
4000,0.032100
5000,0.028600
6000,0.017500
7000,0.014500


TrainOutput(global_step=7779, training_loss=0.04621966293713537, metrics={'train_runtime': 13485.7641, 'train_samples_per_second': 4.615, 'train_steps_per_second': 0.577, 'total_flos': 4346301517052256.0, 'train_loss': 0.04621966293713537, 'epoch': 3.0})

In [20]:
trainer.evaluate()

{'eval_loss': 0.18342378735542297,
 'eval_LOC': {'precision': 0.7900531349628055,
  'recall': 0.8521779000458505,
  'f1': 0.8199404433660527,
  'number': 21810},
 'eval_MISC': {'precision': 0.6147345464458529,
  'recall': 0.6741908961805815,
  'f1': 0.6430914016825894,
  'number': 13379},
 'eval_ORG': {'precision': 0.5614024464091074,
  'recall': 0.5668602873738918,
  'f1': 0.56411816605312,
  'number': 16355},
 'eval_PER': {'precision': 0.8711536534028265,
  'recall': 0.8943252559551371,
  'f1': 0.8825873930593283,
  'number': 19437},
 'eval_overall_precision': 0.7267029169903303,
 'eval_overall_recall': 0.7644299178653442,
 'eval_overall_f1': 0.7450891539132285,
 'eval_overall_accuracy': 0.9696237912048747,
 'eval_runtime': 6503.4528,
 'eval_samples_per_second': 11.796,
 'eval_steps_per_second': 1.475,
 'epoch': 3.0}

In [19]:
model.save_pretrained("./finetuned-bert-conll")
tokenizer.save_pretrained("./finetuned-bert-conll")

('./finetuned-bert-conll\\tokenizer_config.json',
 './finetuned-bert-conll\\special_tokens_map.json',
 './finetuned-bert-conll\\vocab.txt',
 './finetuned-bert-conll\\added_tokens.json',
 './finetuned-bert-conll\\tokenizer.json')

In [15]:
ontoTrainer = Trainer(
    model=model,
    args=args,
    train_dataset=ontonotes_encoded,
    eval_dataset=conll_encoded,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

C:\Users\NIKHIL\AppData\Local\Temp\ipykernel_14528\3048366490.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  ontoTrainer = Trainer(


In [ ]:
ontoTrainer.train()

E:\Software\Anaconda\envs\gpu_env\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Step,Training Loss
1000,0.147800
